In [ ]:
#@title Cell 51.1 — Purpose and fixed scientific analysis plan
from IPython.display import display, Markdown

display(Markdown(r"""
# NB51 — Final BMC comparison with training-only empirical intervals

**Study boundary:** 50,460 independently held-out MIC observations for 9,058 *E. coli* BioSamples and 26 antibiotics, using the original five pathogen-out folds.

**Four interval methods on identical held-out observations:**

1. **BMC-N** — Bayesian normal posterior predictive intervals (NB41).
2. **BMC-T** — Bayesian Student's t intervals with fixed $\nu=4$ (NB42).
3. **BMC-AT** — Bayesian asymmetric Student's t intervals with fixed $\nu=4$ (NB43).
4. **Empirical** — equal-tailed 2.5th and 97.5th percentiles of training-only Model C residuals, estimated *separately for each antibiotic and outer fold*, added to unchanged original Model C predictions.

**Fixed evaluation criteria:** 95% interval coverage, counts and rates above/below limits, mean width, and **95% interval score**, all on the $\log_2(\mathrm{MIC})$ scale. Lower interval score is better because it penalises both width and misses.

$IS_{0.05}(l,u;y)=(u-l)+40(l-y)\mathbf 1(y<l)+40(y-u)\mathbf 1(y>u)$.

**Fair comparison:** Empirical quantiles use only NB40 training-only inner out-of-fold residuals; BMC intervals are reused unchanged. Outer held-out MICs are used **only for evaluation**, never for fitting or method selection. The empirical baseline is not claimed to have an exact finite-sample conformal coverage guarantee.

The 26 antibiotics **do not share a common resistance breakpoint**. Extreme errors and examples are described by residuals, not resistance classifications. No new Model C or BMC models are fitted.

The results are an internal pathogen-out validation and do not constitute external validation. Comparisons on these held-out data are descriptive and should not be used to select a method and claim unbiased performance of that selected method on the same observations.
"""))

In [ ]:
#@title Cell 51.2 — Mount Drive and locate the five validated input files
import os, hashlib, json
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

try:
    from google.colab import drive
    if not Path('/content/drive/MyDrive').is_dir():
        drive.mount('/content/drive')
except ImportError:
    pass

ROOT=Path(os.environ.get('BMC_PROJECT_ROOT','/content/drive/MyDrive/RefMIC'))
if not ROOT.is_dir():
    raise FileNotFoundError(f'RefMIC project folder not found: {ROOT}. Set BMC_PROJECT_ROOT if necessary.')
SEARCH=Path('/content/drive') if Path('/content/drive').is_dir() else ROOT
OUT=ROOT/'03_Results'/'51_BMC_Final_Comparison'
OUT.mkdir(parents=True,exist_ok=True)
SPEC={
    'CAL':('40_BMC_Training_Only_Calibration','40_BMC_training_only_inner_predictions_all_outer_folds.csv.gz'),
    'TEST':('40_BMC_Training_Only_Calibration','40_BMC_independent_outer_test_predictions.csv.gz'),
    'N':('41_BMC_N_Pathogen_Out_Validation','41_BMC_N_independent_outer_test_predictions.csv.gz'),
    'T':('42_BMC_T_Pathogen_Out_Validation','42_BMC_T_independent_outer_test_predictions.csv.gz'),
    'AT':('43_BMC_AT_Pathogen_Out_Validation','43_BMC_AT_independent_outer_test_predictions.csv.gz'),
}

def file_sha256(path):
    d=hashlib.sha256()
    with open(path,'rb') as f:
        for chunk in iter(lambda:f.read(4*1024*1024),b''):
            d.update(chunk)
    return d.hexdigest()

def locate(folder,name):
    preferred=ROOT/'03_Results'/folder/name
    if preferred.is_file():
        return preferred
    matches=[]
    for directory,dirs,files in os.walk(SEARCH):
        dirs[:]=[d for d in dirs if d not in {'.Trashes','.Trash-1000','.git','node_modules','__pycache__'}]
        if name in files:
            matches.append(Path(directory)/name)
    if not matches:
        raise FileNotFoundError(f'Could not find {name} under {SEARCH}. Please verify NB40 and NB41–43 have finished and saved results.')
    if len(matches)>1 and len({file_sha256(p) for p in matches})>1:
        raise ValueError(f'Different copies of {name} exist. Place the authoritative version at {preferred}.\n'+'\n'.join(map(str,matches)))
    return sorted(matches,key=lambda p:(0 if ROOT in p.parents else 1,len(p.parts),str(p)))[0]

PATHS={label:locate(folder,name) for label,(folder,name) in SPEC.items()}
for label,path in PATHS.items():
    print(label,':',path)
print('Results directory:',OUT)

In [ ]:
#@title Cell 51.3 — Load and strictly audit independent pathogen-out observations
KEYS=['interaction_row','biosample','antibiotic','outer_fold']
IDENTITY=KEYS+['observed_log2_mic','model_c_outer_test_predicted_log2_mic']
PREFIX={'N':'bmc_n','T':'bmc_t','AT':'bmc_at'}
cal=pd.read_csv(PATHS['CAL'],compression='gzip')
test=pd.read_csv(PATHS['TEST'],compression='gzip')
cal_need={'outer_fold_for_calibration','inner_fold','interaction_row','biosample','antibiotic','observed_log2_mic','inner_oof_predicted_log2_mic','training_residual_log2'}
if cal_need-set(cal):raise ValueError('NB40 calibration missing columns: '+str(sorted(cal_need-set(cal))))
if set(IDENTITY)-set(test):raise ValueError('NB40 test missing required columns')
if (len(cal),len(test),test.biosample.nunique(),test.antibiotic.nunique())!=(201840,50460,9058,26):
    raise ValueError('Unexpected original Model C data dimensions')
if test.duplicated(['biosample','antibiotic']).any() or test.interaction_row.duplicated().any():
    raise ValueError('Repeated held-out Model C observation')
if test.groupby('biosample').outer_fold.nunique().max()!=1:
    raise ValueError('Held-out BioSample appears in multiple folds')
if set(test.outer_fold)!=set(range(1,6)) or set(cal.outer_fold_for_calibration)!=set(range(1,6)):
    raise ValueError('Original five folds not present')
if cal.duplicated(['outer_fold_for_calibration','interaction_row']).any():
    raise ValueError('Duplicate training observation within an outer fold')
if cal.groupby(['outer_fold_for_calibration','biosample']).inner_fold.nunique().max()!=1:
    raise ValueError('Inner fold changed within a BioSample')
all_original=test.set_index('interaction_row')
for k in range(1,6):
    train=cal.loc[cal.outer_fold_for_calibration.eq(k)]
    held=test.loc[test.outer_fold.eq(k)]
    if len(train)!=len(test)-len(held):raise ValueError(f'Fold {k} wrong training count')
    if set(train.biosample)&set(held.biosample):raise ValueError(f'Fold {k} pathogen leakage')
    if set(train.interaction_row)!=set(test.interaction_row)-set(held.interaction_row):
        raise ValueError(f'Fold {k} training interactions do not match original outer training group')
    original=all_original.loc[train.interaction_row.to_numpy()]
    for name in ['biosample','antibiotic']:
        if not np.array_equal(original[name].astype(str).to_numpy(),train[name].astype(str).to_numpy()):
            raise ValueError(f'Fold {k} {name} identity mismatch')
    if not np.allclose(original.observed_log2_mic,train.observed_log2_mic,rtol=0,atol=1e-9):
        raise ValueError(f'Fold {k} training observed MIC mismatch')
if not np.isfinite(cal[['training_residual_log2','observed_log2_mic','inner_oof_predicted_log2_mic']].to_numpy(float)).all():
    raise ValueError('Nonfinite calibration records')
if not np.allclose(cal.training_residual_log2,cal.observed_log2_mic-cal.inner_oof_predicted_log2_mic,rtol=0,atol=1e-8):
    raise ValueError('Training residual calculation mismatch')

wide=test[IDENTITY].copy()
for model,prefix in PREFIX.items():
    tab=pd.read_csv(PATHS[model],compression='gzip')
    cols=[f'{prefix}_{x}_log2' for x in ['lower95','median','upper95']]
    if set(IDENTITY+cols)-set(tab):raise ValueError(f'BMC-{model} missing interval columns')
    if len(tab)!=50460 or tab.duplicated(KEYS).any():raise ValueError(f'BMC-{model}: unexpected rows')
    chk=wide[IDENTITY].merge(tab[IDENTITY],on=KEYS,validate='one_to_one',how='outer',indicator=True,suffixes=('_reference','_file'))
    if len(chk)!=50460 or not chk._merge.eq('both').all():raise ValueError(f'BMC-{model}: unmatched held-out rows')
    for name in ['observed_log2_mic','model_c_outer_test_predicted_log2_mic']:
        if not np.allclose(chk[f'{name}_reference'],chk[f'{name}_file'],rtol=0,atol=1e-9):
            raise ValueError(f'BMC-{model}: observed outcomes or original Model C predictions differ')
    if not np.isfinite(tab[cols].to_numpy(float)).all():raise ValueError(f'BMC-{model}: nonfinite interval')
    if not ((tab[cols[0]]<=tab[cols[1]])&(tab[cols[1]]<=tab[cols[2]])).all():
        raise ValueError(f'BMC-{model}: invalid predictive interval order')
    wide=wide.merge(tab[KEYS+cols],on=KEYS,how='left',validate='one_to_one')
if wide.isna().any().any():raise ValueError('Missing values after merging all three BMC models')
print(f'Validated {len(wide):,} identical, independent held-out MIC observations for all three BMC models.')
print(f'Validated {len(cal):,} training-only calibration records across 5 overlapping outer training sets.')
print('All MICs remain log2 scale; no shared antibiotic resistance threshold is used.')
display(test.groupby('outer_fold',as_index=False).agg(n_test=('interaction_row','size'),n_pathogens=('biosample','nunique')))

In [ ]:
#@title Cell 51.4 — Form empirical intervals from training-only residual quantiles
Q=[0.025,0.5,0.975]
quantile_rows=[]
for (k,antibiotic),g in cal.groupby(['outer_fold_for_calibration','antibiotic'],sort=True):
    residual=g.training_residual_log2.to_numpy(dtype=float)
    if len(residual)<40:
        raise ValueError(f'Too few calibration residuals for empirical quantiles: fold {k}, {antibiotic}, n={len(residual)}')
    lower,median,upper=np.quantile(residual,Q,method='linear')
    quantile_rows.append({'outer_fold':int(k),'antibiotic':antibiotic,'n_calibration':len(residual),
        'n_calibration_pathogens':g.biosample.nunique(),
        'empirical_residual_lower95_log2':lower,'empirical_residual_median_log2':median,
        'empirical_residual_upper95_log2':upper})
quantiles=pd.DataFrame(quantile_rows)
if len(quantiles)!=5*26 or quantiles.duplicated(['outer_fold','antibiotic']).any():
    raise ValueError('Expected 130 empirical residual calibrations (26 antibiotics × 5 outer folds)')
wide=wide.merge(quantiles[['outer_fold','antibiotic']+[c for c in quantiles if c.startswith('empirical_residual_')]],
                on=['outer_fold','antibiotic'],how='left',validate='many_to_one')
r=wide.model_c_outer_test_predicted_log2_mic
for side in ['lower95','median','upper95']:
    wide[f'empirical_{side}_log2']=r+wide[f'empirical_residual_{side}_log2']
wide=wide.drop(columns=[c for c in wide.columns if c.startswith('empirical_residual_')])
if not np.isfinite(wide[['empirical_lower95_log2','empirical_median_log2','empirical_upper95_log2']].to_numpy(float)).all():
    raise ValueError('Empirical intervals contain nonfinite values')
if not ((wide.empirical_lower95_log2<=wide.empirical_median_log2)&
        (wide.empirical_median_log2<=wide.empirical_upper95_log2)).all():
    raise ValueError('Empirical interval order invalid')
print('Computed 130 antibiotic × outer-fold equal-tailed empirical residual intervals from training-only data.')
print('No held-out residuals were used to estimate these quantiles.')
display(quantiles.loc[quantiles.antibiotic.isin(['ceftazidime','cefepime','ciprofloxacin','meropenem'])].head(12).round(3))

In [ ]:
#@title Cell 51.5 — Final matched comparison: coverage, width, interval score and tail failures
ALPHA=0.05
LABELS=[('BMC-N','bmc_n'),('BMC-T','bmc_t'),('BMC-AT','bmc_at'),('Empirical','empirical')]
BASE=KEYS+['observed_log2_mic','model_c_outer_test_predicted_log2_mic']
parts=[]
for label,prefix in LABELS:
    p=wide[BASE].copy()
    p['method']=label
    for target in ['lower95','median','upper95']:
        p[f'{target}_log2']=wide[f'{prefix}_{target}_log2'].to_numpy(dtype=float)
    yy=p.observed_log2_mic.to_numpy(float)
    ll=p.lower95_log2.to_numpy(float);uu=p.upper95_log2.to_numpy(float)
    p['above_upper95']=yy>uu
    p['below_lower95']=yy<ll
    p['covered_95pct']=(~p.above_upper95)&(~p.below_lower95)
    p['interval_width_log2']=uu-ll
    p['interval_score95_log2']=(uu-ll)+(2/ALPHA)*np.maximum(ll-yy,0)+(2/ALPHA)*np.maximum(yy-uu,0)
    p['distance_above_upper_log2']=np.maximum(yy-uu,0)
    p['point_absolute_error_log2']=np.abs(yy-p.median_log2.to_numpy(float))
    parts.append(p)
long=pd.concat(parts,ignore_index=True)
if len(long)!=4*50460 or not np.isfinite(long[['interval_width_log2','interval_score95_log2','point_absolute_error_log2']].to_numpy(float)).all():
    raise ValueError('Invalid combined held-out scores')
if (long.interval_width_log2<0).any() or (long.interval_score95_log2<0).any():
    raise ValueError('Negative interval widths or scores')

def summarize(g):
    n=len(g)
    return pd.Series({
        'n_test':n,'n_pathogens':g.biosample.nunique(),
        'coverage_95pct':g.covered_95pct.mean(),
        'n_above_upper95':int(g.above_upper95.sum()),'pct_above_upper95':100*g.above_upper95.mean(),
        'n_below_lower95':int(g.below_lower95.sum()),'pct_below_lower95':100*g.below_lower95.mean(),
        'mean_width_log2':g.interval_width_log2.mean(),
        'mean_interval_score95_log2':g.interval_score95_log2.mean(),
        'median_interval_score95_log2':g.interval_score95_log2.median(),
        'mean_upper_exceedance_distance_log2':g.distance_above_upper_log2.mean(),
        'median_point_absolute_error_log2':g.point_absolute_error_log2.median(),
        'point_MAE_log2':g.point_absolute_error_log2.mean(),
    })

overall=long.groupby('method',sort=False).apply(summarize,include_groups=False).reset_index()
by_antibiotic=long.groupby(['method','antibiotic'],sort=False).apply(summarize,include_groups=False).reset_index()
by_fold_antibiotic=long.groupby(['method','outer_fold','antibiotic'],sort=False).apply(summarize,include_groups=False).reset_index()
if len(overall)!=4 or len(by_antibiotic)!=4*26 or len(by_fold_antibiotic)!=4*5*26:
    raise ValueError('Wrong number of summary rows')
if not overall.n_test.eq(50460).all() or not by_antibiotic.groupby('method').n_test.sum().eq(50460).all():
    raise ValueError('Results do not reproduce held-out totals')
original_model_c_mae=np.mean(np.abs(wide.observed_log2_mic-wide.model_c_outer_test_predicted_log2_mic))
print('Overall matched held-out interval comparison (mean interval score: LOWER is better):')
display(overall[['method','n_test','coverage_95pct','n_above_upper95','n_below_lower95','mean_width_log2',
                 'mean_interval_score95_log2','point_MAE_log2']].round(4))
print('Original, unchanged Model C point MAE:',round(original_model_c_mae,4))
print('Focus antibiotics:')
FOCUS=['ceftazidime','cefepime','ciprofloxacin','meropenem']
display(by_antibiotic[by_antibiotic.antibiotic.isin(FOCUS)][
    ['method','antibiotic','n_test','coverage_95pct','pct_above_upper95','mean_width_log2','mean_interval_score95_log2']].round(4))
# The held-out set is a fixed validation set. No automatic choice of a winner occurs here.

In [ ]:
#@title Cell 51.6 — Paired pathogen-cluster bootstrap of interval-score differences
# This quantifies uncertainty in FIXED fitted-model score differences; it does not re-fit models
# and should not be interpreted as an external-validation confidence interval.
N_BOOT=600
rng=np.random.default_rng(51051)
base=wide[['interaction_row','biosample']].copy().set_index('interaction_row')
scored={label:long.loc[long.method.eq(label),['interaction_row','interval_score95_log2']]
        .set_index('interaction_row').rename(columns={'interval_score95_log2':label}) for label,_ in LABELS}
scores=base.copy()
for label,_ in LABELS:
    scores=scores.join(scored[label],how='left',validate='one_to_one')
if scores.isna().any().any() or len(scores)!=50460:raise ValueError('Unmatched scores for bootstrap')
per_path=scores.groupby('biosample').agg(n=('biosample','size'),**{label:(label,'sum') for label,_ in LABELS})
if len(per_path)!=9058:raise ValueError('Expected 9,058 independent pathogen clusters')
arr=per_path[['n']+[label for label,_ in LABELS]].to_numpy(dtype=float)
indices=rng.integers(0,len(arr),size=(N_BOOT,len(arr)))
boot_n=arr[indices,0].sum(axis=1)
base_index=[label for label,_ in LABELS].index('Empirical')+1
boot_base=arr[indices,base_index].sum(axis=1)/boot_n
boot_rows=[]
for i,(label,_) in enumerate(LABELS):
    if label=='Empirical':continue
    boot_score=arr[indices,i+1].sum(axis=1)/boot_n
    improvement=boot_base-boot_score
    actual=(scores['Empirical']-scores[label]).mean()
    low,hi=np.quantile(improvement,[0.025,0.975])
    boot_rows.append({'method':label,'comparator':'Empirical',
        'observed_mean_interval_score_improvement_log2':actual,
        'bootstrap_2p5pct':low,'bootstrap_97p5pct':hi,
        'fraction_bootstrap_improvement_gt_0':float((improvement>0).mean()),
        'n_bootstrap':N_BOOT,'cluster':'biosample'})
bootstrap=pd.DataFrame(boot_rows)
print('Positive difference = lower (better) interval score than the empirical baseline.')
display(bootstrap.round(4))
print('Bootstrap re-samples BioSamples with all associated MIC observations together; models remain fixed.')

In [ ]:
#@title Cell 51.7 — Final figures: coverage, interval score and upper-tail failures
import matplotlib.ticker as mtick
# Default matplotlib colours; identical method order across figures.
METHODS=[label for label,_ in LABELS]
ant_order=(by_antibiotic.loc[by_antibiotic.method.eq('Empirical')]
            .sort_values('n_test',ascending=False).antibiotic.tolist())
Y=np.arange(len(ant_order));offsets=[-.27,-.09,.09,.27]
fig,ax=plt.subplots(figsize=(11,10))
for (label,_),d in zip(LABELS,offsets):
    vals=by_antibiotic[by_antibiotic.method.eq(label)].set_index('antibiotic').loc[ant_order,'coverage_95pct'].to_numpy()*100
    ax.plot(vals,Y+d,'o',markersize=4,label=label)
ax.axvline(95,ls='--',lw=1,color='gray',label='Nominal 95%')
ax.set_yticks(Y,ant_order);ax.invert_yaxis();ax.set_xlabel('Observed 95% interval coverage (%)')
ax.set_title('Figure 51.1  |  Held-out interval coverage by antibiotic')
ax.grid(axis='x',alpha=.2);ax.legend(ncol=5,fontsize=8,loc='upper center',bbox_to_anchor=(.5,-.055))
fig.tight_layout();fig.savefig(OUT/'51_Figure1_26_Antibiotic_Coverage.png',dpi=180,bbox_inches='tight');plt.show();plt.close(fig)

fig,ax=plt.subplots(figsize=(11,10))
for label,d in zip(METHODS,offsets):
    vals=by_antibiotic[by_antibiotic.method.eq(label)].set_index('antibiotic').loc[ant_order,'mean_interval_score95_log2'].to_numpy()
    ax.plot(vals,Y+d,'o',markersize=4,label=label)
ax.set_yticks(Y,ant_order);ax.invert_yaxis();ax.set_xlabel('Mean 95% interval score (log₂ MIC; lower is better)')
ax.set_title('Figure 51.2  |  Paired held-out interval scores by antibiotic')
ax.grid(axis='x',alpha=.2);ax.legend(ncol=4,fontsize=8,loc='upper center',bbox_to_anchor=(.5,-.055))
fig.tight_layout();fig.savefig(OUT/'51_Figure2_26_Antibiotic_Interval_Score.png',dpi=180,bbox_inches='tight');plt.show();plt.close(fig)

fig,ax=plt.subplots(figsize=(11,10))
for label,d in zip(METHODS,offsets):
    vals=by_antibiotic[by_antibiotic.method.eq(label)].set_index('antibiotic').loc[ant_order,'pct_above_upper95'].to_numpy()
    ax.plot(vals,Y+d,'o',markersize=4,label=label)
ax.set_yticks(Y,ant_order);ax.invert_yaxis();ax.set_xlabel('Observed MICs above the upper 95% limit (%)')
ax.set_title('Figure 51.3  |  Upper-tail exceedances by antibiotic')
ax.grid(axis='x',alpha=.2);ax.legend(ncol=4,fontsize=8,loc='upper center',bbox_to_anchor=(.5,-.055))
fig.tight_layout();fig.savefig(OUT/'51_Figure3_26_Antibiotic_Upper_Exceedances.png',dpi=180,bbox_inches='tight');plt.show();plt.close(fig)

fig,ax=plt.subplots(figsize=(7.4,5))
for row in overall.itertuples(index=False):
    ax.scatter(row.mean_width_log2,100*row.coverage_95pct,s=95,label=row.method)
    ax.annotate(row.method,(row.mean_width_log2,100*row.coverage_95pct),xytext=(7,5),textcoords='offset points',fontsize=9)
ax.axhline(95,ls='--',lw=1,color='gray',label='Nominal 95%')
ax.set_xlabel('Mean interval width (log₂ MIC)');ax.set_ylabel('Observed coverage (%)')
ax.set_title('Figure 51.4  |  Coverage versus interval width (all held-out MICs)')
ax.grid(alpha=.2);fig.tight_layout();fig.savefig(OUT/'51_Figure4_Overall_Coverage_Width.png',dpi=180,bbox_inches='tight');plt.show();plt.close(fig)
print('Saved four manuscript-ready comparison figures to',OUT)

In [ ]:
#@title Cell 51.8 — Same 25 highest Model C underpredictions: 4 methods, 4 antibiotics
# Extreme examples are chosen ONLY for plotting from the held-out original Model C errors,
# not for fitting the predictive distributions or empirical residual baseline.
for antibiotic in FOCUS:
    d=wide.loc[wide.antibiotic.eq(antibiotic)].copy()
    d['original_underprediction_log2']=d.observed_log2_mic-d.model_c_outer_test_predicted_log2_mic
    d=d.nlargest(25,'original_underprediction_log2').sort_values('original_underprediction_log2')
    ypos=np.arange(len(d)); shifts=[-.27,-.09,.09,.27]
    fig,ax=plt.subplots(figsize=(11,10))
    for (label,prefix),shift in zip(LABELS,shifts):
        lo=d[f'{prefix}_lower95_log2'].to_numpy();hi=d[f'{prefix}_upper95_log2'].to_numpy()
        mid=d[f'{prefix}_median_log2'].to_numpy()
        (line,)=ax.plot([],[],lw=2,label=label)
        col=line.get_color()
        ax.hlines(ypos+shift,lo,hi,color=col,lw=2,alpha=.8)
        ax.plot(mid,ypos+shift,'o',markersize=3,color=col)
    ax.plot(d.observed_log2_mic.to_numpy(),ypos,'kx',markersize=6,mew=1.4,label='Observed MIC')
    ax.set_yticks(ypos,d.biosample.tolist(),fontsize=8)
    ax.set_xlabel('log₂(MIC)');ax.set_ylabel('Held-out BioSample')
    ax.set_title(f'Figure 51.5  |  {antibiotic}: 25 largest original Model C underpredictions')
    ax.grid(axis='x',alpha=.2);ax.legend(ncol=5,loc='upper center',bbox_to_anchor=(.5,-.055),fontsize=8)
    fig.tight_layout()
    fig.savefig(OUT/f'51_Figure5_Extreme_Cases_{antibiotic.replace(" ","_")}.png',dpi=180,bbox_inches='tight')
    plt.show();plt.close(fig)
print('These focus figures are illustrative subsets, not independent evidence of overall performance.')

In [ ]:
#@title Cell 51.9 — Export final tables, provenance audit, and freeze coding results
files={
'51_Final_Overall_Comparison.csv':overall,
'51_Final_26_Antibiotic_Comparison.csv':by_antibiotic,
'51_Final_Fold_Antibiotic_Comparison.csv':by_fold_antibiotic,
'51_Training_Only_Empirical_Quantiles.csv':quantiles,
'51_Paired_Pathogen_Bootstrap_Interval_Score.csv':bootstrap,
}
for filename,df in files.items():
    df.to_csv(OUT/filename,index=False)
    print('Saved:',filename,'(',len(df),'rows )')
long.to_csv(OUT/'51_All_Four_Methods_Independent_Heldout_Intervals.csv.gz',index=False,compression='gzip')
print('Saved: 51_All_Four_Methods_Independent_Heldout_Intervals.csv.gz')

# Provenance and hard invariants: the four reported methods retain the same original observed MICs.
ids=long.groupby(['interaction_row','biosample','antibiotic','outer_fold']).agg(
    n_methods=('method','nunique'),n_observed_values=('observed_log2_mic','nunique'),
    n_model_c_values=('model_c_outer_test_predicted_log2_mic','nunique'))
if len(ids)!=50460 or (ids.n_methods!=4).any() or (ids.n_observed_values!=1).any() or (ids.n_model_c_values!=1).any():
    raise ValueError('Final four-way held-out identity audit failed')
if not (overall.n_above_upper95+overall.n_below_lower95+overall.coverage_95pct*overall.n_test).round().eq(overall.n_test).all():
    raise ValueError('Final coverage audit failed')
audit={
    'notebook':'NB51 Final BMC interval comparison v0.1',
    'models':['BMC-N','BMC-T','BMC-AT','Empirical'],
    'n_observations':50460,'n_unique_biosamples':9058,'n_antibiotics':26,
    'n_outer_folds':5,'n_calibration_records_across_overlapping_training_sets':201840,
    'empirical_interval_quantiles':[0.025,0.5,0.975],
    'empirical_interval_source':'NB40 inner cross-fitted Model C training residuals per outer fold and antibiotic',
    'interval_alpha':ALPHA,
    'interval_score':'width + 2/alpha*(lower-y)_+ + 2/alpha*(y-upper)_+',
    'validation':'Same original pathogen-out held-out MIC observations for all four models; no held-out MIC used for interval calibration',
    'limitation':'Internal validation, not independent external validation; comparisons are descriptive and do not account for post-hoc method selection',
    'input_files':{name:{'path':str(path),'sha256':file_sha256(path)} for name,path in PATHS.items()},
    'output_files':{},
}
for path in sorted(OUT.iterdir()):
    if path.is_file() and path.name!='51_Provenance_And_Validation.json':
        audit['output_files'][path.name]={'sha256':file_sha256(path),'bytes':path.stat().st_size}
(OUT/'51_Provenance_And_Validation.json').write_text(json.dumps(audit,indent=2),encoding='utf-8')
print('\nNB51 complete. All four methods compared on identical independent held-out observations.')
print('Outputs:',OUT)
print('The computational workflow is ready to freeze for manuscript preparation; no method was selected using the held-out results.')
display(overall[['method','coverage_95pct','mean_width_log2','mean_interval_score95_log2','n_above_upper95']].round(4))